# Spread souverains et risque politique en zone euro

Ces dernières années, la question de la dette publique et de la santé financière des États est revenue au cœur de l'actualité économique, en particulier au sein de la zone euro. Avec la fin des politiques de taux d'intérêt négatifs, le durcissement de la politique monétaire et les crises budgétaires successives, les marchés financiers scrutent de très près la viabilité des finances publiques. L'accentuation rapide et marquée des écarts de taux s'explique généralement par la combinaison d'une instabilité politique, de déficits budgétaires persistants et d'un niveau élevé d'endettement public. Lorsque ces facteurs se cumulent, les investisseurs craignent une perte de contrôle budgétaire ou un risque de non-soutenabilité, ce qui peut provoquer une réévaluation brutale du risque, une fuite vers des actifs plus sûrs et une envolée des coûts de financement pour les États concernés.

Pour mesurer et comparer la santé financière de ces pays, les économistes et les marchés utilisent un indicateur clé : le spread souverain. Très simplement, le spread correspond à l'écart de taux d'intérêt entre l'obligation d'un État à 10 ans et celle de l'Allemagne, qui sert de référence « sans risque » dans la zone euro. Il se calcule par une simple soustraction :


$$\text{Spread} = \text{Taux du pays} - \text{Taux de l'Allemagne}$$


Si la France emprunte à 3 % et l'Allemagne à 2 %, le spread français est de 1 % (soit 100 points de base). Plus ce chiffre est élevé, plus les marchés estiment que le pays présente un risque, ce qui alourdit le coût de financement de sa dette.

De nombreuses études académiques se sont penchées sur ces dynamiques de marché et sur la formation de ces écarts de taux. Les travaux de **Paul De Grauwe et Yuemei Ji (2012)**, notamment sur la fragilité inhérente de la zone euro, montrent que les spreads ne dépendent pas seulement des fondamentaux économiques d'un pays, mais peuvent aussi fluctuer sous l'effet de mouvements de panique auto-réalisateurs des marchés. D'autres recherches, comme celles de **Kerstin Bernoth, Jürgen von Hagen et Ludger Schuknecht (2012)** sur la tarification du risque souverain, soulignent quant à elles le rôle central de la dette et des déficits dans l'évaluation du risque par les investisseurs. 

Notre projet s'inscrit dans cette lignée : en nous concentrant sur la France, l'Allemagne, l'Italie et l'Espagne à partir de 2016, nous chercherons à observer et analyser l'évolution de ces spreads souverains au fil du temps.


# PARTIE 1 : Collecte et nettoyage des données

## A. Importation des données :

### 0. Installation


In [ ]:
from __future__ import annotations

import json
from datetime import date
from pathlib import Path

import pandas as pd
import requests

### 1. Préparation de l'importation
Pour structurer l'analyse des spreads souverains, l'étude se focalise sur quatre pays de la zone euro : la France, l'Allemagne, l'Italie et l'Espagne (`COUNTRIES = ["FR", "DE", "IT", "ES"]`), à partir du 1er janvier 2016 (`START_DATE`).

Nous créons `RAW DIR` afin de stocker les données brutes et éviter des requêtes répétées lors de la relance du script.

Des en-têtes personnalisés (`HEADERS`) incluant un User-Agent sont configurés pour identifier proprement le script auprès des serveurs et éviter d'éventuels blocages, associés à un `TIMEOUT` de 60 secondes.


In [ ]:
# Définition des chemins d'accès aux données 
ROOT = Path(__file__).resolve().parents[1]
DATA_DIR = ROOT / "data"
COUNTRIES = ["FR", "DE", "IT", "ES"]  # Périmètre d'étude : France, Allemagne, Italie, Espagne 
RAW_DIR = DATA_DIR / "raw"            # Dossier dédié au stockage des données brutes 
START_DATE = pd.Timestamp("2016-01-01")

# Paramètres des requêtes HTTP pour l'importation des données
HEADERS = {"User-Agent": "Mozilla/5.0 (projet ENSAE - spreads souverains)"}
TIMEOUT = 60

### 2. Origine des données 

Pour récupérer l'historique des taux des quatre pays (France, Allemagne, Italie, Espagne) depuis le 1er janvier 2016, nous centralisons trois flux de données directement en ligne :

- **CNBC** (`CNBC_URL`) : Récupération des taux journaliers (les cours de clôture) via l'API du site, grâce à des codes propres à chaque pays (`CNBC_SYMBOLS`).
- **Bundesbank** (`BUNDESBANK_URL`) : Téléchargement direct des taux allemands à 10 ans, qui servent de taux de référence ("sans risque") pour toute la zone euro.
- **Eurostat** (`EUROSTAT_UR`) : Téléchargement d'un fichier (au format `SDMX-CSV`) contenant les taux mensuels de convergence à long terme de l'Union européenne.

Il est important de noter que CNBC est notre source principale. Bindesbank et Eurostat sont des sources officielles qui nous permettrons de vérifier les données de CNBC.

In [ ]:
# Définition des URLs pour l'importation des données
CNBC_SYMBOLS = {c: f"{c}10Y-{c}" for c in COUNTRIES}
CNBC_URL = (
    "https://ts-api.cnbc.com/harmony/app/bars/{symbol}/1D/"
    "{start}000000/{end}000000/adjusted/EST5EDT.json"
)
BUNDESBANK_URL = (
    "https://api.statistiken.bundesbank.de/rest/download/BBSIS/"
    "D.I.ZST.ZI.EUR.S1311.B.A604.R10XX.R.A.A._Z._Z.A?format=csv&lang=en"
)
EUROSTAT_URL = (
    "https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/"
    "irt_lt_mcby_m/M.MCBY.{geos}?format=SDMX-CSV&startPeriod={start}"
)

La fonction `_get(url : str)` nous permet de récuperer les fichiers bruts en évitant des bugs réseau.

In [ ]:
def _get(url: str) -> requests.Response:
    response = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
    response.raise_for_status()  # Lève une exception si le statut HTTP indique une erreur
    return response

### 3. Données journalières par pays 
La fonction `fetch_cnbc_country` récupére les cours de clôture quotidiens des obligations souveraines à 10 ans pour chaque pays de notre périmètre, en passant par l'API de CNBC.

Cette fonction s'appuie sur notre système de cache local (`RAW_DIR`): si le fichier n'a pas encore été téléchargé, une requête est envoyée à l'API et le fichier brut est stocké au format `JSON`. Le script lit ensuite ce fichier pour en extraire les prix et les dates, avant de structurer le tout sous forme de série temporelle propre et triée chronologiquement.

In [ ]:
def fetch_cnbc_country(country: str, refresh: bool = False) -> pd.Series:
    """Rendement à 10 ans journalier (clôture, en %) d'un pays."""
    cache = RAW_DIR / f"cnbc_{country}10Y.json"
    if refresh or not cache.exists():  # Mécanisme de cache : on ne télécharge que si nécessaire
        url = CNBC_URL.format(
            symbol=CNBC_SYMBOLS[country],
            start=START_DATE.strftime("%Y%m%d"),
            end=date.today().strftime("%Y%m%d"),
        )
        cache.write_text(_get(url).text)

    bars = json.loads(cache.read_text())["barData"]["priceBars"] 
    series = pd.Series(
        {pd.Timestamp(bar["tradeTime"][:8]): float(bar["close"]) for bar in bars},
        name=country,
    ) 
    return series.sort_index()

La fonction `fetch_cnbc` permet de regrouper les séries individuelles de chaque pays dans un tableau unifié.

Pour ce faire, le script utilise la fonction `pd.concat` afin d'aligner côte à côte (en colonnes via l'argument `axis=1`) les séries de taux obtenues pour l'ensemble des pays du périmètre. L'index du tableau final est ensuite explicitement nommé date afin de garantir une structuration homogène et exploitable sous forme de DataFrame.


In [ ]:
def fetch_cnbc(refresh: bool = False) -> pd.DataFrame:
    """Tableau (dates x pays) des rendements à 10 ans CNBC, en %."""
    frame = pd.concat(
        [fetch_cnbc_country(c, refresh=refresh) for c in COUNTRIES], axis=1
    )  # Concaténation des séries de chaque pays en colonnes
    frame.index.name = "date"
    return frame

### 4. Benchmark allemand

La fonction `fetch_bundesbank`récupère les emprunts fédéraux allemands à 10 ans. 

Cette fonction récupère les taux mensuels de long terme d'Eurostat via l'API, puis transforme les données brutes d'un format long à un format large pour que chaque pays apparaisse en colonne.

In [ ]:
def fetch_bundesbank(refresh: bool = False) -> pd.Series:
    """Taux à 10 ans des titres fédéraux allemands (Svensson), en %."""
    cache = RAW_DIR / "bundesbank_10y.csv"
    if refresh or not cache.exists():
        cache.write_bytes(_get(BUNDESBANK_URL).content)

    raw = pd.read_csv(
        cache,
        skiprows=9,
        header=None,
        names=["date", "value", "flag"],
        na_values=["."],
        encoding="utf-8-sig",
    )  # Lecture du CSV en ignorant l'en-tête technique
    raw["date"] = pd.to_datetime(raw["date"], errors="coerce")  # Nettoyage des dates invalides
    series = raw.dropna(subset=["date", "value"]).set_index("date")["value"]
    series = pd.to_numeric(series, errors="coerce").dropna()  # Conversion propre en numérique
    return series.loc[START_DATE:].rename("DE_bundesbank")


### 5. Taux mensuels de convergence de l'Union Européenne 

La fonction `fetch_eurostat_monthly`récupère les taux mensuels de convergence de l'Union Européenne publiés par Eurostat sous forme de moyennes mensuelles (`irt_lt_mcby_m`). 

La fonction récupère les données ia l'API puis les transforme le tableau brut d'un format long à un format large pour que les périodes forment l'index et que les pays apparaissent en colonnes.

In [ ]:
def fetch_eurostat_monthly(refresh: bool = False) -> pd.DataFrame:
    """Taux de convergence à 10 ans (moyennes mensuelles), en %."""
    cache = RAW_DIR / "eurostat_irt_lt_mcby_m.csv"
    if refresh or not cache.exists():
        url = EUROSTAT_URL.format(
            geos="+".join(COUNTRIES), start=START_DATE.strftime("%Y-%m")
        )
        cache.write_bytes(_get(url).content)

    raw = pd.read_csv(cache)
    frame = raw.pivot(index="TIME_PERIOD", columns="geo", values="OBS_VALUE")  # Passage du format long au large
    frame.index = pd.PeriodIndex(frame.index, freq="M")
    frame.index.name = "mois" 
    return frame[COUNTRIES]

### 6. Collecte des données 

Enfin, le bloc principal du script s'assure d'abord que l'arborescence des dossiers de stockage brut (raw) est bien créée (`RAW_DIR.mkdir`), puis déclenche séquentiellement l'exécution des fonctions d'importation. Le paramètre `refresh = False` est configuré par défaut pour exploiter le cache local.

In [ ]:
RAW_DIR.mkdir(parents=True, exist_ok=True)  # Crée l'arborescence des dossiers si elle n'existe pas

refresh = False
fetch_cnbc(refresh)  # "refresh=True" permet de forcer le téléchargement si nécessaire
fetch_bundesbank(refresh)
fetch_eurostat_monthly(refresh)

##  B. Nettoyage des taux souverains à 10 ans (FR, DE, IT, ES)

**Objectif:** Partir des données brutes téléchargées par `imporationdata.py` (dossier `raw/`) pour obtenir des séries propres, alignées et comparables entre pays, enregistrées dans `cleaned/`.

**Démarche,** pour chaque source :
1. on affiche les données brutes
2. on repère ce qui pose problème
3. on corrige
4. on vérifie le résultat

Les fichiers sont lus depuis `raw/`, copiés et modifiés par les fonctions de `cleaning.py` puis la version "propre" se trouve dans `cleaned/`.

### 0. Préparation

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Le notebook est dans data/. Si on le lance depuis la racine du projet, on descend dans data/.
if (Path.cwd() / "raw").exists():
    DATA_DIR = Path.cwd()
else:
    DATA_DIR = Path.cwd() / "data"
RAW_DIR = DATA_DIR / "raw"

sys.path.append(str(DATA_DIR))  # pour pouvoir importer cleaning.py
import cleaning

COUNTRIES = cleaning.COUNTRIES
DAY_NAMES = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]

plt.rcParams["figure.figsize"] = (11, 4)

print("Fichiers bruts :", [p.name for p in RAW_DIR.iterdir()])
print("Début de la période d'étude :", cleaning.START_DATE.date())

### 1. CNBC : rendements journaliers à 10 ans

#### 1.1 Le fichier brut

L'API CNBC renvoie un fichier JSON par pays. Chaque « barre » correspond à une journée de cotation.

In [ ]:
path = RAW_DIR / "cnbc_FR10Y.json"
bars = json.loads(path.read_text())["barData"]["priceBars"]

print(f"{len(bars)} barres pour la France. Les 3 premières :")
for bar in bars[:3]:
    print(bar)

In [ ]:
for country in COUNTRIES:
    path = RAW_DIR / f"cnbc_{country}10Y.json"
    bars = json.loads(path.read_text())["barData"]["priceBars"]
    first_date = bars[0]["tradeTime"][:8]
    last_date = bars[-1]["tradeTime"][:8]
    print(f"{country} : {len(bars)} barres, du {first_date} au {last_date}")

print("Type du champ 'close' :", type(bars[0]["close"]))

**Dans le fichier brut :**
- `close` est le rendement de clôture en %, mais il est stocké comme texte (`"0.9890"`) : nous le convertirons à un objet `float`.
- `tradeTime` est une date au format `AAAAMMJJhhmmss` : on garde les 8 premiers caractères et on le tansforme en un objet `date`.
- On ignore les colonnes `volume`, `open`, `high` et `low`.
- Les pays n'ont pas le même nombre de barres sur la même période : leurs calendriers ne sont pas alignés, on va compléter les valeurs manquantes par pays.

`cleaning.load_cnbc()` regroupe les quatre pays dans un seul tableau (dates en lignes, pays en colonnes). Cette étape **met en forme** les données, sans encore rien supprimer.

In [ ]:
cnbc_raw = cleaning.load_cnbc()
print("Dimensions :", cnbc_raw.shape)
cnbc_raw.head(10)

In [ ]:
print("Nombre de valeurs manquantes par pays :")
cnbc_raw.isna().sum()

In [ ]:
cnbc_raw.plot(title="CNBC : rendements à 10 ans BRUTS (%) par pays", linewidth=0.8)
plt.ylabel("%")
plt.show()

À l'œil, les courbes semblent cohérentes, mais le tableau contient déjà des `NaN` dès les premières lignes : le 2 et le 3 janvier 2016 sont un samedi et un dimanche.

#### 1.2 Problème n°1 : des cotations le week-end

In [ ]:
# Exemple sur une semaine
week = cnbc_raw.loc["2016-01-07":"2016-01-12"].copy()
week["jour"] = [DAY_NAMES[d] for d in week.index.dayofweek]
week

On remarque qu'il y a certain week-ends avec les valeurs complétées, surtout le samedi, d'autres non. De plus elles n'existent pas pour tous les pays le même jour.

Cependant, les marchés de dette souveraine de la zone euro sont fermés le week-end. Ces valeurs ne sont donc pas de vraies clôtures : ce sont des cotations indicatives et peu liquides. Les garder aurait trois effets :
- Fausses variations : un taux qui bouge un samedi augmente artificiellement la volatilité ;
- Pays décalés : un spread FR − DE calculé un samedi comparerait une cotation hors séance à une valeur manquante ;
- Calendrier irrégulier : une semaine aurait 5, 6 ou 7 observations selon le pays.

Donc, on ne garde que les jours du lundi au vendredi (`dayofweek < 5`), à partir du début de la période d'étude.

In [ ]:
before = cnbc_raw.loc[cleaning.START_DATE:]
step1 = before[before.index.dayofweek < 5]

print(f"Lignes avant : {len(before)}, après : {len(step1)}")
print(f"→ {len(before) - len(step1)} lignes de week-end supprimées")
print("Samedis ou dimanches restants :", (step1.index.dayofweek >= 5).sum())

#### 1.3 Problème n°2 : des jours ouvrés manquants

Quand on met les pays côte à côte, un jour absent pour un pays apparaît comme un `NaN`. Mais un jour absent pour tous les pays disparaît complètement du tableau.

On a donc recréé le calendrier complet des jours ouvrés avec `pd.bdate_range`, puis on aligne le tableau dessus avec `reindex`. Les jours absents deviennent des lignes de `NaN`, visibles.

In [ ]:
all_business_days = pd.bdate_range(step1.index.min(), step1.index.max())
print(f"Jours ouvrés du calendrier : {len(all_business_days)}")
print(f"Jours présents dans les données : {len(step1)}")

absent_for_all = [d.date() for d in all_business_days if d not in step1.index]
print("Jours absents pour TOUS les pays :", absent_for_all)

step2 = step1.reindex(all_business_days)

In [ ]:
incomplete = step2[step2.isna().any(axis=1)].copy()
incomplete["jour"] = [DAY_NAMES[d] for d in incomplete.index.dayofweek]
print(f"{len(incomplete)} jours ouvrés où au moins un pays manque :")
incomplete

In [ ]:
# Longueur des trous : rolling(n).sum() == n signifie "n valeurs manquantes d'affilée"
for country in COUNTRIES:
    missing = step2[country].isna()
    n_two = (missing.rolling(2).sum() == 2).sum()
    n_three = (missing.rolling(3).sum() == 3).sum()
    print(f"{country} : {missing.sum()} jours manquants, {n_two} fois 2 jours d'affilée, {n_three} fois 3 jours d'affilée")

**Diagnostic :** Les trous sont rares et courts (1 ou 2 jours, jamais 3). Ils tombent surtout sur des jours fériés nationaux (Noël et le 26 décembre, lundi de Pâques, 1er mai…).

**Solution :** Nous avons décidé de reporter la dernière valeur (`ffill`). Cela suppose que le taux n'a pas bougé pendant le jour manquant, mais cela reste cohérent avec un marché fermé, et n'utilise que le passé

On limite le report à 2 jours (`ffill(limit=2)`). Au-delà, supposer un taux constant devient trop fort : on préfère alors supprimer la ligne.

**Contrepartie :** un jour comblé a une variation nulle, ce qui sous-estime un tout petit peu la volatilité. L'effet est négligeable vu le petit nombre de jours concernés.


#### 1.5 Résultat

In [ ]:
cnbc, cnbc_filled = cleaning.clean_yields(cnbc_raw)
cleaning.check_daily(cnbc, cnbc_filled, "CNBC 10 ans")
cnbc.describe().round(2)

In [ ]:
cnbc.plot(title="CNBC : rendements à 10 ans NETTOYÉS (%)", linewidth=0.8)
plt.ylabel("%")
plt.show()


On obtient un calendrier régulier de jours ouvrés, sans valeur manquante, où les 4 pays sont observés les mêmes jours. Un spread compare donc toujours deux valeurs du même jour.

### 2. Bundesbank : taux allemand à 10 ans

#### 2.1 Le fichier brut

In [ ]:
lines = (RAW_DIR / "bundesbank_10y.csv").read_text(encoding="utf-8-sig").splitlines()

print("Début du fichier :")
for line in lines[:13]:
    print(line)
print("...")
print("Fin du fichier :")
for line in lines[-3:]:
    print(line)

**Ce qu'on lit dans le fichier brut :**
- les 9 premières lignes sont des métadonnées: on va les sauter avec `skiprows=9` ;
- une valeur manquante est notée `.,No value available` : on la lit comme `NaN` avec `na_values=["."]` ;
- le fichier contient tous les jours depuis 1997, week-ends compris, on va restreindre.

On commence par charger le fichier sous la bonne forme avec `cleaning.load_bundesbank()`.

In [ ]:
bb_raw = cleaning.load_bundesbank()
bb_period = bb_raw.loc[cleaning.START_DATE:]
bb_period.head(10)

Analyse de la localisation des données manquantes :

In [ ]:
share_missing = bb_period["DE_bundesbank"].isna().groupby(bb_period.index.dayofweek).mean()
share_missing.index = DAY_NAMES
print("Part de valeurs manquantes selon le jour :")
share_missing.round(3)

**Diagnostic.** Les samedis et dimanches sont **toujours** vides, et quelques jours ouvrés aussi (jours fériés allemands). C'est le même problème que pour CNBC. On regarde la longueur des trous sur les jours ouvrés :

In [ ]:
bb_bdays = bb_period[bb_period.index.dayofweek < 5]
bb_bdays = bb_bdays.reindex(pd.bdate_range(bb_bdays.index.min(), bb_bdays.index.max()))

missing = bb_bdays["DE_bundesbank"].isna()
three_in_a_row = missing.rolling(3).sum() == 3
print(f"{missing.sum()} jours ouvrés manquants")
print("Trous de 3 jours, qui se terminent le :", [d.date() for d in missing[three_in_a_row].index])

pd.DataFrame({
    "avant": bb_bdays.loc["2018-12-20":"2018-12-28", "DE_bundesbank"],
    "après ffill": bb_bdays.loc["2018-12-20":"2018-12-28", "DE_bundesbank"].ffill(limit=cleaning.FFILL_LIMIT),
})

La Bundesbank ne publie pas les jours fériés allemands, d'où plus de jours à combler que pour CNBC. Quelques années ont un trou de **3 jours** (du 24 au 26 décembre). Avec `ffill(limit=2)`, le 24 et le 25 reçoivent la valeur du 21, mais le 26 reste vide et sa ligne est supprimée : au-delà de 2 jours, on n'invente pas de valeur. Le 1er janvier 2016 est aussi supprimé, car aucune valeur antérieure ne permet de le combler.

#### 2.2 Correction et résultat

On applique **la même fonction `clean_yields`**, avec les mêmes règles, pour que les sources soient comparables.

In [ ]:
bundesbank, bundesbank_filled = cleaning.clean_yields(bb_raw)
cleaning.check_daily(bundesbank, bundesbank_filled, "Bundesbank DE 10 ans")

bundesbank.plot(title="Bundesbank : taux allemand à 10 ans NETTOYÉ (%)", linewidth=0.8, legend=False)
plt.ylabel("%")
plt.show()

### 3. Eurostat : taux de convergence mensuels

#### 3.1 Le fichier brut

In [ ]:
eu_csv = pd.read_csv(RAW_DIR / "eurostat_irt_lt_mcby_m.csv")
print("Dimensions :", eu_csv.shape)
eu_csv.head()

**Ce qu'on lit dans le fichier brut :**
- le fichier a un format particulier : une ligne par couple (pays, mois). Pour comparer les pays, il faut le changer à un format mois en lignes, pays en colonnes : c'est ce que fait `pivot` ;
- plusieurs colonnes ont une seule valeur (`DATAFLOW`, `freq`, `int_rt`…) et n'apportent rien ;
- la donnée est mensuelle, donc pas de problème de week-end. Il reste à vérifier qu'aucun mois ne manque.

#### 3.2 Correction et résultat

In [ ]:
eurostat = cleaning.clean_monthly(cleaning.load_eurostat())
cleaning.check_monthly(eurostat, "Eurostat 10 ans (mensuel)")
eurostat.head()

In [ ]:
eurostat_plot = eurostat.copy()
eurostat_plot.index = pd.to_datetime(eurostat_plot.index)  # "2016-01" -> date, pour le graphique
eurostat_plot.plot(title="Eurostat : taux de convergence à 10 ans (%)")
plt.ylabel("%")
plt.show()

### 5. Enregistrement dans `cleaned/`

On lance le script complet (`cleaning.main()`), comme avec `python data/cleaning.py` dans un terminal. Il refait les contrôles puis écrit les trois fichiers.

In [ ]:
print("\nRelecture des fichiers enregistrés :")
for path in (DATA_DIR / "cleaned").glob("*.csv"):
    df = pd.read_csv(path, index_col=0)
    print(f"- {path.name} : {df.shape[0]} lignes, {df.shape[1]} colonnes, de {df.index[0]} à {df.index[-1]}")

**Limites :**
- Le calendrier est celui du lundi au vendredi : un jour férié où une source publie quand même une valeur (par exemple le 1er janvier) est conservé.
- Avec `ffill(limit=2)`, un trou de 3 jours ou plus voit ses 2 premiers jours comblés et les suivants supprimés. Les trous de CNBC ne dépassent pas 2 jours ; côté Bundesbank, cela ne concerne que quelques 26 décembre.
- Les jours comblés ont une variation nulle, ce qui sous-estime légèrement la volatilité.
- CNBC n'est pas une source officielle : il faut désormais s'occuper de la comparaison entre ses sources pour vérifier sa fiabilité.